In [ ]:
# GPU-Optimized Synthetic Data Generation for Kaggle
# Generates 200 images per minority class: CIN1, CIN2, CIN3, Cancer
# Estimated time: 2-3 hours total on GPU

# Install dependencies
%pip install -q diffusers transformers accelerate torch pillow

import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

print('✓ Setup complete!')
print('✓ Running on Kaggle with GPU')

In [ ]:
# Improved Synthetic Data Generator with Better Prompts
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import os
from tqdm import tqdm

class SyntheticDataGenerator:
    def __init__(self, model_id="runwayml/stable-diffusion-v1-5"):
        print(f"Loading Stable Diffusion model: {model_id}")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
        
        self.pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            safety_checker=None,
            requires_safety_checker=False
        )
        self.pipe = self.pipe.to(device)
        
        if device == "cuda":
            self.pipe.enable_attention_slicing()
            print("✓ Attention slicing enabled for memory efficiency")
        
        # Enhanced prompts for better quality
        self.prompts = {
            'CIN1': "histopathology microscopy image, epithelial tissue with mild cellular changes, dysplastic cells, medical slide, scientific photograph, diagnostic pathology",
            'CIN2': "histopathology microscopy image, epithelial tissue with moderate cellular abnormalities, dysplastic tissue, medical slide, scientific photograph, diagnostic pathology",
            'CIN3': "histopathology microscopy image, epithelial tissue with severe cellular changes, high-grade dysplasia, medical slide, scientific photograph, diagnostic pathology",
            'Cancer': "histopathology microscopy image, malignant epithelial cells, invasive carcinoma tissue, medical slide, scientific photograph, diagnostic pathology",
            'Normal': "histopathology microscopy image, pink epithelial cells, normal tissue structure, medical slide, scientific photograph, clinical diagnostic image"
        }
        
        self.negative_prompt = "blurry, low quality, cartoon, illustration, text, watermark, signature, person, face, body"
    
    def generate_images(self, stage, num_images, output_dir, start_idx=1):
        """
        Generate synthetic images
        
        Args:
            stage: Class name (CIN1, CIN2, CIN3, Cancer, Normal)
            num_images: Number of images to generate
            output_dir: Directory to save images
            start_idx: Starting index for filenames (useful for adding to existing images)
        """
        os.makedirs(output_dir, exist_ok=True)
        prompt = self.prompts.get(stage, self.prompts['Normal'])
        
        print(f"\n{'='*80}")
        print(f"Generating {num_images} images for {stage}")
        print(f"Output: {output_dir}")
        print(f"{'='*80}")
        print(f"Prompt: {prompt}\n")
        
        generated_files = []
        
        for i in tqdm(range(num_images), desc=f"Generating {stage}"):
            image = self.pipe(
                prompt,
                negative_prompt=self.negative_prompt,
                num_inference_steps=50,
                guidance_scale=7.5,
                height=512,
                width=512
            ).images[0]
            
            filename = f"{stage}_{start_idx + i}.png"
            filepath = os.path.join(output_dir, filename)
            image.save(filepath)
            generated_files.append(filepath)
        
        print(f"\n✓ Generated {num_images} images for {stage}")
        print(f"Files: {stage}_{start_idx}.png to {stage}_{start_idx + num_images - 1}.png")
        return generated_files

print('✓ Generator class loaded!')

In [ ]:
# Initialize generator (this takes 2-3 minutes)
generator = SyntheticDataGenerator()

# Create output directories (Kaggle uses /kaggle/working/)
!mkdir -p /kaggle/working/synthetic_images/CIN1
!mkdir -p /kaggle/working/synthetic_images/CIN2
!mkdir -p /kaggle/working/synthetic_images/CIN3
!mkdir -p /kaggle/working/synthetic_images/Cancer

print('\n✓ Generator initialized with GPU!')
print('Ready to generate images!')

In [ ]:
# Generate 200 images for each minority class
# Total: 800 images, ~2-3 hours on GPU

# CIN1 (200 images, ~30 min)
generator.generate_images(
    stage='CIN1',
    num_images=200,
    output_dir='/kaggle/working/synthetic_images/CIN1',
    start_idx=1
)

# CIN2 (200 images, ~30 min)
generator.generate_images(
    stage='CIN2',
    num_images=200,
    output_dir='/kaggle/working/synthetic_images/CIN2',
    start_idx=1
)

# CIN3 (200 images, ~30 min)
generator.generate_images(
    stage='CIN3',
    num_images=200,
    output_dir='/kaggle/working/synthetic_images/CIN3',
    start_idx=1
)

# Cancer (200 images, ~30 min)
generator.generate_images(
    stage='Cancer',
    num_images=200,
    output_dir='/kaggle/working/synthetic_images/Cancer',
    start_idx=1
)

print('\n' + '='*80)
print('✓ All images generated!')
print('='*80)
print('Summary:')
print(f'  CIN1: 200 images')
print(f'  CIN2: 200 images')
print(f'  CIN3: 200 images')
print(f'  Cancer: 200 images')
print(f'  Total: 800 new images')

In [ ]:
# Verify generated images
import os

for stage in ['CIN1', 'CIN2', 'CIN3', 'Cancer']:
    folder = f'/kaggle/working/synthetic_images/{stage}'
    count = len([f for f in os.listdir(folder) if f.endswith('.png')])
    print(f'{stage}: {count} images')

In [ ]:
# Zip all generated images for download
!cd /kaggle/working && zip -r synthetic_minority_images.zip synthetic_images/

print('✓ Created synthetic_minority_images.zip')
print('File size:')
!ls -lh /kaggle/working/synthetic_minority_images.zip

In [ ]:
# Download instructions for Kaggle
print('✓ Generation complete!')
print('='*80)
print('The zip file is saved at: /kaggle/working/synthetic_minority_images.zip')
print('='*80)
print('\nTo download:')
print('1. Click on "Data" tab in the right sidebar')
print('2. Navigate to "Output" section')
print('3. Click the download button next to synthetic_minority_images.zip')
print('\nThis contains 800 images (200 each for CIN1, CIN2, CIN3, Cancer)')
print('Extract and copy to your training folders!')

---
## Download from Kaggle:

1. **Click "Data" tab** in the right sidebar
2. **Navigate to "Output"** section
3. **Click download** next to `synthetic_minority_images.zip`

## Next Steps on Your Mac:

```bash
cd ~/Downloads
unzip synthetic_minority_images.zip

# Copy to training folders
cp synthetic_images/CIN1/*.png /Users/akshitarora/cervical-cancer-classifier-local/data/train/CIN1/
cp synthetic_images/CIN2/*.png /Users/akshitarora/cervical-cancer-classifier-local/data/train/CIN2/
cp synthetic_images/CIN3/*.png /Users/akshitarora/cervical-cancer-classifier-local/data/train/CIN3/
cp synthetic_images/Cancer/*.png /Users/akshitarora/cervical-cancer-classifier-local/data/train/Cancer/
```

## New Training Data Totals:
- **CIN1**: ~150 → 350 images (+200)
- **CIN2**: ~111 → 311 images (+200)
- **CIN3**: ~111 → 311 images (+200)
- **Cancer**: ~111 → 311 images (+200)
- **Normal**: ~164 images (unchanged)

## Expected Results After Retraining:
With Focal Loss + More Synthetic Images:
- **Cancer recall**: 9% → **45-55%** ✓
- **CIN2 recall**: 36% → **50-60%** ✓
- **CIN3 recall**: 27% → **45-55%** ✓
- **Overall accuracy**: 72% → **78-82%** ✓

## Time Estimate:
- Image generation: **2-3 hours** on Kaggle GPU
- Download + copy: **5 minutes**
- Retrain with Focal Loss: **45-60 minutes** on GPU
- Total: **3-4 hours**